It is highly recommended to use a powerful **GPU**, you can use it for free uploading this notebook to [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb).
<table align="center">
 <td align="center"><a target="_blank" href="https://colab.research.google.com/github/ezponda/intro_deep_learning/blob/main/class/CNN/Depth_Estimation.ipynb">
        <img src="https://colab.research.google.com/img/colab_favicon_256px.png"  width="50" height="50" style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/Depth_Estimation.ipynb">
        <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"  width="50" height="50" style="padding-bottom:5px;" />View Source on GitHub</a></td>
</table>

**Table of Contents**

1. [Introduction to Monocular Depth Estimation](#introduction)
2. [Depth Anything V2](#depth-anything-v2)
3. [Basic Inference](#basic-inference)
4. [Comparing Model Sizes](#comparing-model-sizes)
5. [Depth-Based Visual Effects](#depth-based-effects)
    * 5.1 [Portrait Mode (Bokeh Effect)](#bokeh)
    * 5.2 [Fog Effect](#fog)
6. [3D Point Cloud Visualization](#point-cloud)
7. [Combining Object Detection and Depth](#yolo-depth)

<a id='introduction'></a>
# Introduction to Monocular Depth Estimation

**Monocular depth estimation** is the task of predicting the distance of every pixel in an image from the camera, using only a single 2D photograph. Unlike stereo cameras or LiDAR sensors that use multiple viewpoints or active sensing, monocular depth relies entirely on visual cues learned by neural networks.

Humans are naturally good at estimating depth from a single image: we use cues like object size, occlusion, texture gradients, and perspective. Deep learning models learn similar cues from large datasets.

### Applications

Monocular depth estimation is widely used in:

- **Smartphone photography**: Portrait mode / bokeh effect (iPhone, Pixel) uses depth to blur the background
- **Autonomous driving**: Estimating distance to obstacles from camera feeds (Tesla's camera-only approach)
- **Augmented Reality**: Placing virtual objects at the correct depth in real scenes (Meta Quest, Apple Vision Pro)
- **Robotics and drones**: Obstacle avoidance using cameras instead of expensive LiDAR
- **3D reconstruction**: Creating 3D models from photographs (NeRF, Gaussian Splatting)

<a id='depth-anything-v2'></a>
# Depth Anything V2

[Depth Anything V2](https://arxiv.org/abs/2406.09414) (NeurIPS 2024) is a state-of-the-art monocular depth estimation model. It uses a **DINOv2 Vision Transformer** as encoder and a **DPT (Dense Prediction Transformer)** decoder to produce dense per-pixel depth maps.

### How It Works

The model is trained in three stages:

1. **Synthetic teacher**: A large model (ViT-Giant) is trained on ~600K synthetic images with pixel-perfect ground truth depth. Synthetic data avoids the noise and gaps present in real-world depth sensors (LiDAR, stereo).

2. **Pseudo-labeling**: The teacher generates depth labels for 62 million real-world images, bridging the synthetic-to-real gap.

3. **Student training**: Smaller, faster student models (Small, Base, Large) are trained on these pseudo-labeled real images.

This strategy produces sharp, detailed depth maps that handle challenging cases like transparent objects, reflections, and thin structures.

### Model Variants

| Model | Parameters | Speed (T4 GPU) | Quality |
|-------|-----------|----------------|---------|
| Small | 24.8M | ~30-50 ms | Good |
| Base | 97.5M | ~70-120 ms | Better |
| Large | 335.3M | ~150-250 ms | Best |

All models are available on [HuggingFace](https://huggingface.co/depth-anything).

In [ ]:
#!pip install transformers accelerate
#!pip install plotly
#!pip install ultralytics

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline
import cv2
import urllib

<a id='basic-inference'></a>
# Basic Inference

Let's load the smallest model and run it on an image. The output is a **relative depth map**: higher values mean closer to the camera, lower values mean further away.

In [ ]:
# Load the depth estimation pipeline (Small model for speed)
depth_pipe = pipeline(task="depth-estimation",
                      model="depth-anything/Depth-Anything-V2-Small-hf",
                      device=0 if torch.cuda.is_available() else -1)

# Download a sample image
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/traffic.jpg'
image_path = 'traffic.jpg'
urllib.request.urlretrieve(url, image_path)

image = Image.open(image_path).convert("RGB")
print(f"Image size: {image.size}")

# Run depth estimation
result = depth_pipe(image)
depth_map = result["depth"]  # PIL Image (grayscale)
depth_array = np.array(depth_map)

print(f"Depth map shape: {depth_array.shape}")
print(f"Depth range: [{depth_array.min()}, {depth_array.max()}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(image)
axes[0].set_title('Original Image')
axes[0].axis('off')

im = axes[1].imshow(depth_array, cmap='inferno')
axes[1].set_title('Depth Map (brighter = closer)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, label='Relative depth')

plt.tight_layout()
plt.show()

### Exploring Different Colormaps

The depth map is a single-channel image. Different colormaps highlight different aspects of the depth information:

In [ ]:
colormaps = ['inferno', 'viridis', 'plasma', 'magma', 'gray']

fig, axes = plt.subplots(1, len(colormaps), figsize=(20, 4))
for ax, cmap in zip(axes, colormaps):
    ax.imshow(depth_array, cmap=cmap)
    ax.set_title(cmap)
    ax.axis('off')
plt.suptitle('Depth map with different colormaps', fontsize=14)
plt.tight_layout()
plt.show()

### Try with different images

Let's see how the model performs on different types of scenes:

In [ ]:
image_urls = {
    'Street': 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/traffic.jpg',
    'Portrait': 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/albert_einstein.jpg',
    'Beach': 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/beach.jpg',
}

fig, axes = plt.subplots(len(image_urls), 2, figsize=(12, 5 * len(image_urls)))

for row, (scene_name, url) in enumerate(image_urls.items()):
    img_path = f'{scene_name.lower()}.jpg'
    urllib.request.urlretrieve(url, img_path)
    img = Image.open(img_path).convert("RGB")
    depth = depth_pipe(img)["depth"]
    
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'{scene_name} - Original')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(np.array(depth), cmap='inferno')
    axes[row, 1].set_title(f'{scene_name} - Depth')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

<a id='comparing-model-sizes'></a>
# Comparing Model Sizes

Depth Anything V2 comes in three sizes. Let's compare speed and quality:

In [ ]:
import time

models = {
    "Small (24.8M)":  "depth-anything/Depth-Anything-V2-Small-hf",
    "Base (97.5M)":   "depth-anything/Depth-Anything-V2-Base-hf",
    "Large (335.3M)": "depth-anything/Depth-Anything-V2-Large-hf",
}

image = Image.open('traffic.jpg').convert("RGB")
results = {}

for name, model_id in models.items():
    pipe = pipeline(task="depth-estimation", model=model_id,
                    device=0 if torch.cuda.is_available() else -1)
    
    # Warmup
    _ = pipe(image)
    
    # Timed inference (average of 5 runs)
    start = time.time()
    for _ in range(5):
        result = pipe(image)
    elapsed = (time.time() - start) / 5
    
    results[name] = {"time_ms": elapsed * 1000, "depth": np.array(result["depth"])}
    print(f"{name}: {elapsed*1000:.1f} ms/image")

# Side-by-side comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis('off')

for i, (name, data) in enumerate(results.items(), 1):
    axes[i].imshow(data["depth"], cmap='inferno')
    axes[i].set_title(f'{name}\n{data["time_ms"]:.0f} ms')
    axes[i].axis('off')

plt.suptitle('Model size comparison: speed vs quality', fontsize=14)
plt.tight_layout()
plt.show()

<a id='depth-based-effects'></a>
# Depth-Based Visual Effects

Now that we can estimate depth, we can use it to create visual effects that depend on distance from the camera.

<a id='bokeh'></a>
## Portrait Mode (Bokeh Effect)

Smartphone cameras simulate the shallow depth-of-field of professional cameras by blurring the background while keeping the subject sharp. This requires knowing which pixels are close (subject) and which are far (background).

The idea is simple:
1. Estimate the depth map
2. Create a mask separating foreground from background
3. Blur the background
4. Composite the sharp foreground over the blurred background

In [ ]:
# Reload the small model for the rest of the notebook
depth_pipe = pipeline(task="depth-estimation",
                      model="depth-anything/Depth-Anything-V2-Small-hf",
                      device=0 if torch.cuda.is_available() else -1)

def apply_bokeh(image_pil, depth_pil, blur_strength=51, threshold=0.5):
    """Simulate portrait mode by blurring pixels beyond a depth threshold."""
    img = np.array(image_pil)
    depth = np.array(depth_pil.convert("L")).astype(np.float32) / 255.0
    
    # Higher values = closer in Depth Anything V2
    # Create smooth mask: 1 for foreground (close), 0 for background (far)
    mask = cv2.GaussianBlur(
        (depth > threshold).astype(np.float32), (21, 21), 0
    )
    
    # Blur the entire image
    blurred = cv2.GaussianBlur(img, (blur_strength, blur_strength), 0)
    
    # Composite: sharp foreground + blurred background
    result = (img * mask[:, :, None] + blurred * (1 - mask[:, :, None])).astype(np.uint8)
    return Image.fromarray(result)


# Apply to portrait image
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/albert_einstein.jpg'
urllib.request.urlretrieve(url, 'portrait.jpg')
portrait = Image.open('portrait.jpg').convert("RGB")

depth_result = depth_pipe(portrait)
depth_map = depth_result["depth"]

bokeh = apply_bokeh(portrait, depth_map, blur_strength=51, threshold=0.45)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(portrait)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(np.array(depth_map), cmap='inferno')
axes[1].set_title('Depth Map')
axes[1].axis('off')

axes[2].imshow(bokeh)
axes[2].set_title('Portrait Mode (Bokeh)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Question 1: Adjust the Bokeh Effect

Experiment with the `threshold` and `blur_strength` parameters of `apply_bokeh`. Try different images (street scene, beach) and find settings that produce a natural-looking result.

- What happens when you increase the threshold?
- What happens when you increase the blur strength?

In [ ]:
# Try the bokeh effect on a different image with different parameters
image = Image.open('traffic.jpg').convert("RGB")
depth_map = depth_pipe(image)["depth"]

bokeh = apply_bokeh(image, depth_map, blur_strength=..., threshold=...)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(bokeh)
axes[1].set_title('Bokeh Effect')
axes[1].axis('off')
plt.tight_layout()
plt.show()

<a id='fog'></a>
## Fog Effect

We can simulate atmospheric fog that increases with distance. This is physically accurate: objects further from the camera are more obscured by fog.

In [ ]:
def apply_fog(image_pil, depth_pil, fog_color=(220, 230, 240), fog_intensity=0.85):
    """Apply distance-based fog: more fog for distant objects."""
    img = np.array(image_pil).astype(np.float32)
    depth = np.array(depth_pil.convert("L")).astype(np.float32) / 255.0
    
    # Invert: far pixels get more fog (low depth = far = high fog)
    fog_factor = (1 - depth) * fog_intensity
    fog_layer = np.array(fog_color, dtype=np.float32)
    
    result = img * (1 - fog_factor[:, :, None]) + fog_layer * fog_factor[:, :, None]
    return Image.fromarray(np.clip(result, 0, 255).astype(np.uint8))


image = Image.open('traffic.jpg').convert("RGB")
depth_map = depth_pipe(image)["depth"]
foggy = apply_fog(image, depth_map)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(np.array(depth_map), cmap='inferno')
axes[1].set_title('Depth Map')
axes[1].axis('off')

axes[2].imshow(foggy)
axes[2].set_title('Fog Effect')
axes[2].axis('off')

plt.tight_layout()
plt.show()

<a id='point-cloud'></a>
# 3D Point Cloud Visualization

We can convert a 2D image and its depth map into a 3D point cloud using the **pinhole camera model**. Each pixel is back-projected into 3D space based on its depth value.

The back-projection equations are:

$$X = \frac{(u - c_x) \cdot Z}{f}, \quad Y = \frac{(v - c_y) \cdot Z}{f}$$

where $(u, v)$ are pixel coordinates, $(c_x, c_y)$ is the image center, $f$ is the focal length, and $Z$ is the depth.

In [ ]:
import plotly.graph_objects as go

def depth_to_point_cloud(image_pil, depth_pil, focal_length=500.0, max_points=50000):
    """Convert image + depth map to a colored 3D point cloud."""
    img = np.array(image_pil)
    depth = np.array(depth_pil.convert("L")).astype(np.float32)
    
    H, W = depth.shape
    cx, cy = W / 2, H / 2
    
    # Create pixel coordinate grid
    u = np.arange(W)
    v = np.arange(H)
    uu, vv = np.meshgrid(u, v)
    
    # Back-project to 3D
    Z = depth / 255.0 * 10  # Scale to ~10 meters
    X = (uu - cx) * Z / focal_length
    Y = (vv - cy) * Z / focal_length
    
    # Flatten and subsample for performance
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    colors = img.reshape(-1, 3)
    
    idx = np.random.choice(len(pts), min(max_points, len(pts)), replace=False)
    return pts[idx], colors[idx]


image = Image.open('traffic.jpg').convert("RGB")
depth_map = depth_pipe(image)["depth"]

pts, colors = depth_to_point_cloud(image, depth_map)

fig = go.Figure(data=[go.Scatter3d(
    x=pts[:, 0], y=pts[:, 2], z=-pts[:, 1],
    mode='markers',
    marker=dict(
        size=1.5,
        color=[f'rgb({r},{g},{b})' for r, g, b in colors],
    )
)])
fig.update_layout(
    title="3D Point Cloud from a Single Photo",
    scene=dict(aspectmode='data'),
    width=900, height=700
)
fig.show()

## Question 2: 3D Point Cloud from a Different Image

Generate a 3D point cloud from one of the other images (portrait, beach). Experiment with the `focal_length` parameter:
- What happens with a larger focal length?
- What happens with a smaller one?
- Which value produces the most realistic-looking 3D reconstruction?

In [ ]:
image = Image.open(...).convert("RGB")
depth_map = depth_pipe(image)["depth"]

pts, colors = depth_to_point_cloud(image, depth_map, focal_length=...)

fig = go.Figure(data=[go.Scatter3d(
    x=pts[:, 0], y=pts[:, 2], z=-pts[:, 1],
    mode='markers',
    marker=dict(size=1.5,
                color=[f'rgb({r},{g},{b})' for r, g, b in colors])
)])
fig.update_layout(scene=dict(aspectmode='data'), width=900, height=700)
fig.show()

<a id='yolo-depth'></a>
# Combining Object Detection and Depth

By combining YOLO (object detection) with Depth Anything V2, we can answer questions like: **which detected object is closest to the camera?**

This is useful in robotics, autonomous driving, and AR applications where knowing both *what* an object is and *how far* it is matters.

In [ ]:
from ultralytics import YOLO
import matplotlib.patches as patches

# Load YOLO
yolo = YOLO('yolo11n.pt')

def detect_and_rank_by_depth(image_path, depth_pipeline, yolo_model):
    """Detect objects with YOLO and rank them by estimated depth."""
    image = Image.open(image_path).convert("RGB")
    
    # Depth estimation
    depth_result = depth_pipeline(image)
    depth_array = np.array(depth_result["depth"].convert("L")).astype(np.float32) / 255.0
    
    # Object detection
    detections = yolo_model(np.array(image), verbose=False)[0]
    
    objects = []
    for box in detections.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_name = detections.names[int(box.cls[0])]
        conf = float(box.conf[0])
        
        # Median depth within bounding box (more robust than center pixel)
        roi_depth = depth_array[y1:y2, x1:x2]
        median_depth = np.median(roi_depth) if roi_depth.size > 0 else 0
        
        objects.append({
            "class": cls_name, "confidence": conf,
            "box": (x1, y1, x2, y2), "depth": median_depth
        })
    
    # Sort by depth (highest = closest)
    objects.sort(key=lambda x: x["depth"], reverse=True)
    return objects, depth_array, image


objects, depth_array, image = detect_and_rank_by_depth('traffic.jpg', depth_pipe, yolo)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(image)
cmap = plt.cm.RdYlGn
for i, obj in enumerate(objects):
    x1, y1, x2, y2 = obj["box"]
    color = cmap(1 - i / max(len(objects) - 1, 1))
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor='none')
    axes[0].add_patch(rect)
    rank = "CLOSEST" if i == 0 else f"#{i+1}"
    axes[0].text(x1, y1-5, f'{rank}: {obj["class"]}',
                 color='white', fontsize=8, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
axes[0].set_title('Objects ranked by distance (green = closest)')
axes[0].axis('off')

axes[1].imshow(depth_array, cmap='inferno')
axes[1].set_title('Depth Map')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Objects sorted by closeness:")
for i, obj in enumerate(objects):
    print(f"  #{i+1} {obj['class']} -- depth: {obj['depth']:.3f} (conf: {obj['confidence']:.2f})")

## Question 3: Depth-Based Background Blur with Object Detection

Combine YOLO detection with the bokeh effect: instead of using a fixed depth threshold, use YOLO to detect a person and blur everything that is further away.

Steps:
1. Run YOLO to detect a person and get their bounding box
2. Run depth estimation on the image
3. Compute the median depth of the detected person
4. Use that depth value as the threshold for `apply_bokeh`

In [ ]:
image = Image.open('portrait.jpg').convert("RGB")

# Step 1: Detect objects with YOLO
detections = yolo(np.array(image), verbose=False)[0]

# Step 2: Estimate depth
depth_result = depth_pipe(image)
depth_map = depth_result["depth"]
depth_array = np.array(depth_map.convert("L")).astype(np.float32) / 255.0

# Step 3: Find the person's depth
# Hint: iterate over detections.boxes, find class 'person',
# compute median depth in their bounding box
person_depth = ...

# Step 4: Apply bokeh using the person's depth as threshold
# Hint: use apply_bokeh with threshold slightly below person_depth
bokeh = apply_bokeh(image, depth_map, blur_strength=..., threshold=...)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(bokeh)
axes[1].set_title('Smart Bokeh (blur behind person)')
axes[1].axis('off')
plt.tight_layout()
plt.show()